# Food Delivery Operations & Customer Analytics

## 1. Business Problem

Food delivery businesses generate large amounts of transactional and customer data.
However, raw data alone does not provide clear insights into customer behavior,
order performance, revenue, discounts, delivery charges, and operational issues.

The objective of this project is to analyze food delivery order data and identify
patterns that can help the business understand its customers, monitor order
performance, evaluate discount usage, and identify potential operational issues.

The analysis will use Python and Pandas to perform data quality checks, data
cleaning, exploratory analysis, and business-focused analysis.

## 2. Project Objectives

The key objectives of this project are:

1. Understand the structure and quality of the food delivery dataset.
2. Identify missing, inconsistent, and potentially incorrect data.
3. Clean and prepare the data for analysis.
4. Analyze order value, delivery fees, discounts, and final amounts.
5. Identify patterns in order status and customer behavior.
6. Analyze the impact of discounts on orders and revenue.
7. Identify operational issues and unusual data points.
8. Generate business insights and recommendations from the analysis.

## 3. Dataset Overview

The dataset contains food delivery order-level information.

Each row represents an individual food delivery order.

The dataset contains information related to:

- Order identification
- Customer identification
- Restaurant identification
- Order date and time
- Order status
- Food amount
- Delivery fee
- Discount
- Final amount
- Payment method

The dataset will first be inspected for its structure, data types, missing values,
duplicates, and potential data quality issues before performing further analysis.

## 4. Import Libraries

In [63]:
import pandas as pd
import matplotlib.pyplot as plt

## 5. Load Dataset

The raw food delivery order data is loaded into a Pandas DataFrame for data quality assessment and further analysis.

The initial data audit will focus on:

- Dataset structure and dimensions
- Column names and data types
- Missing values
- Potential outliers
- Inconsistent or invalid values

In [64]:
orders = pd.read_csv('/content/orders.csv')
orders.head()

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method
0,ORD0000001,CUST00613,REST0160,2024-07-03,16:44:28,Delivered,3227.95,34.74,484.19,2778.50,Debit Card
1,ORD0000002,CUST02175,REST0218,2022-06-12,00:48:39,Cancelled,409.02,16.19,40.90,384.31,Wallet
2,ORD0000003,CUST00419,REST0115,2023-06-08,03:06:36,Delivered,654.29,45.23,98.14,601.38,UPI
3,ORD0000004,CUST01177,REST0129,2024-02-04,14:05:22,Cancelled,771.58,28.63,0.00,800.21,Debit Card
4,ORD0000005,CUST01215,REST0021,2023-04-21,07:27:52,Delivered,195.51,10.00,19.55,185.96,UPI


## 6. Initial Data Inspection

Before performing data cleaning, the dataset is inspected to understand its structure, dimensions, data types, missing values, and potential data quality issues.

The initial inspection helps determine which fields require further validation or correction before analysis.

### 6.1 Dataset Dimensions

The number of rows and columns is examined to understand the size of the dataset.

In [65]:
orders.shape

(60060, 11)

The dataset contains 60060 food delivery orders and 11 columns.

### 6.2 Column Information and Data Types

The structure of each column is reviewed to understand the available fields, data types, and presence of missing values.

In [66]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60060 entries, 0 to 60059
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order_ID        60060 non-null  object 
 1   Customer_ID     60060 non-null  object 
 2   Restaurant_ID   60060 non-null  object 
 3   Order_Date      60060 non-null  object 
 4   Order_Time      60060 non-null  object 
 5   Order_Status    60060 non-null  object 
 6   Food_Amount     60060 non-null  float64
 7   Delivery_Fee    60060 non-null  float64
 8   Discount        59610 non-null  float64
 9   Final_Amount    60060 non-null  float64
 10  Payment_Method  60060 non-null  object 
dtypes: float64(4), object(7)
memory usage: 5.0+ MB


### Observation

* Order_Date is stored as str
* Discount has missing values
* Numeric fields are stored as float64
* There are 60,060 records

### 6.3 Statistical Summary

Descriptive statistics are reviewed for numerical columns to understand the distribution of the data and identify unusual or potentially extreme values.

In [67]:
orders.describe()

,Food_Amount,Delivery_Fee,Discount,Final_Amount
count,60060.000000,60060.000000,59610.000000,60060.000000
mean,487.658985,35.040403,41.776208,481.507484
std,288.194958,11.810303,47.902853,265.433759
min,-1165.680000,10.000000,0.000000,78.000000
25%,289.620000,26.850000,0.000000,299.027500
50%,419.510000,34.980000,29.980000,418.155000
75%,608.420000,43.070000,60.517500,592.130000
max,3500.000000,85.630000,600.000000,3516.380000


### 6.4 Missing Value Analysis

Missing values are identified to determine whether any columns require data validation or correction before further analysis.

The analysis will identify:

- Columns containing missing values
- Number of missing records
- Percentage of missing records
- Whether the missing values can be logically validated or reconstructed

In [68]:
orders.isna().sum()

,0
Order_ID,0
Customer_ID,0
Restaurant_ID,0
Order_Date,0
Order_Time,0
Order_Status,0
Food_Amount,0
Delivery_Fee,0
Discount,450
Final_Amount,0


### Observation

* All columns have 0 missing values except Discount.
* Discount has 450 missing values.
* Total records = 60,060.
* Therefore, missing Discount values = 450 / 60,060 ≈ 0.75%.

### 6.5 Missing Value Percentage

The percentage of missing values is calculated to understand the significance of the missing records relative to the total dataset.

The Discount column contains missing values, so its missing-value percentage will be calculated separately.

In [69]:
discount_missing_percentage = (orders['Discount'].isna().sum() / len(orders))*100
print(round(discount_missing_percentage,2),'%')

0.75 %


### Observation

The Discount column contains 450 missing values out of 60,060 records, representing approximately 0.75% of the dataset.

Although the percentage of missing values is relatively small, the records require further investigation because Discount is used in the calculation of Final Amount.

The missing Discount values will therefore be validated using the relationship between Food Amount, Delivery Fee, and Final Amount.

### 6.6 Validate the missing discount

In [70]:
orders['Calculated_Discount'] = (orders['Food_Amount'] + orders['Delivery_Fee']) - orders['Final_Amount']
display(orders.head())
display(orders.tail())

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount
0,ORD0000001,CUST00613,REST0160,2024-07-03,16:44:28,Delivered,3227.95,34.74,484.19,2778.50,Debit Card,484.19
1,ORD0000002,CUST02175,REST0218,2022-06-12,00:48:39,Cancelled,409.02,16.19,40.90,384.31,Wallet,40.90
2,ORD0000003,CUST00419,REST0115,2023-06-08,03:06:36,Delivered,654.29,45.23,98.14,601.38,UPI,98.14
3,ORD0000004,CUST01177,REST0129,2024-02-04,14:05:22,Cancelled,771.58,28.63,0.00,800.21,Debit Card,0.00
4,ORD0000005,CUST01215,REST0021,2023-04-21,07:27:52,Delivered,195.51,10.00,19.55,185.96,UPI,19.55


,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount
60055,ORD0013291,CUST00974,REST0003,2023-11-17,10:47:20,Delivered,283.63,33.41,42.54,274.50,UPI,42.54
60056,ORD0057607,CUST00521,REST0200,2022-09-07,02:13:40,Delivered,488.77,33.43,48.88,473.32,UPI,48.88
60057,ORD0047190,CUST00144,REST0150,2022-08-14,18:52:39,Delivered,1047.03,48.17,0.00,1095.20,UPI,0.00
60058,ORD0029210,CUST00196,REST0149,2023-12-30,20:42:35,Delivered,203.41,38.67,30.51,211.57,Credit Card,30.51
60059,ORD0042188,CUST00980,REST0021,2024-12-13,22:52:19,Delivered,794.52,37.57,79.45,752.64,Wallet,79.45


### Observation

Discount is missing in 450 records. Since Final Amount, Food Amount and Delivery Fee are available, the missing Discount can potentially be reconstructed using the relationship between these fields.


In [71]:
orders[orders['Discount'].isna()][['Food_Amount' , 'Delivery_Fee' , 'Discount' , 'Final_Amount' , 'Calculated_Discount']]

,Food_Amount,Delivery_Fee,Discount,Final_Amount,Calculated_Discount
64,519.30,43.45,NaN,484.85,77.90
294,330.25,34.08,NaN,298.28,66.05
311,1162.82,48.13,NaN,1094.67,116.28
348,410.46,28.80,NaN,377.69,61.57
400,270.59,13.94,NaN,257.47,27.06
...,...,...,...,...,...
59669,319.15,40.63,NaN,343.82,15.96
59759,131.98,45.75,NaN,157.93,19.80
59824,878.28,35.36,NaN,913.64,0.00
59867,709.36,35.40,NaN,638.36,106.40


### 6.7 Replace the Missing Discount

In [72]:
orders['Discount'] = orders['Discount'].fillna(orders['Calculated_Discount'])
display(orders.isna().sum())

display(orders.iloc[400,8])

,0
Order_ID,0
Customer_ID,0
Restaurant_ID,0
Order_Date,0
Order_Time,0
Order_Status,0
Food_Amount,0
Delivery_Fee,0
Discount,0
Final_Amount,0


np.float64(27.059999999999945)

### Observation

The missing Discount values were replaced with the corresponding Calculated_Discount values, while existing Discount values were retained. The missing-value count was then verified to be zero.

Also, randomly checked one missing discount value is replaced by the correct calculated discount value (index number 400)


### 6.8 Duplicate Record Analysis

Duplicate records can lead to double-counting of orders, revenue, discounts, and other business metrics.

The dataset will therefore be checked for completely duplicated records before proceeding with further analysis.

The analysis will determine:

Whether duplicate rows exist
The number of duplicate records
Whether the duplicates represent genuine repeated transactions or duplicate data entries

In [73]:
print(orders.duplicated().sum())


60


In [74]:
display(orders[orders.duplicated()])

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount
60000,ORD0012629,CUST00479,REST0017,2023-07-05,22:12:18,Cancelled,435.07,36.78,21.75,450.10,Wallet,2.175000e+01
60001,ORD0037731,CUST00505,REST0168,2022-01-30,02:05:57,Delivered,172.03,46.23,17.20,201.06,Wallet,1.720000e+01
60002,ORD0039992,CUST01190,REST0002,2024-03-14,04:37:01,Delivered,585.06,19.54,87.76,516.84,UPI,8.776000e+01
60003,ORD0008526,CUST00689,REST0021,2022-04-07,11:06:24,Delivered,767.17,39.42,0.00,806.59,UPI,-1.136868e-13
60004,ORD0008280,CUST02427,REST0002,2022-01-29,02:37:21,Delivered,663.98,56.29,33.20,687.07,UPI,3.320000e+01
60005,ORD0051013,CUST02160,REST0225,2023-05-22,12:14:48,Delivered,356.19,38.96,53.43,341.72,Credit Card,5.343000e+01
60006,ORD0014872,CUST01958,REST0188,2024-09-15,03:32:51,Delivered,309.62,44.06,15.48,338.20,Debit Card,1.548000e+01
60007,ORD0015128,CUST00435,REST0166,2023-06-15,22:16:24,Delivered,460.12,40.24,46.01,454.35,UPI,4.601000e+01
60008,ORD0009367,CUST00568,REST0101,2024-05-24,16:05:52,Delivered,787.74,31.55,118.16,701.13,Debit Card,1.181600e+02
60009,ORD0033323,CUST00283,REST0206,2023-12-04,14:02:15,Delivered,194.20,52.88,0.00,247.08,Debit Card,-2.842171e-14


In [75]:
print(orders['Order_ID'].duplicated().sum())

60


In [76]:
orders = orders.drop_duplicates()
print(orders.duplicated().sum())

0


### Observation

The dataset initially contained 60,060 records. The duplicate analysis identified 60 exact duplicate records. The Order_ID validation also identified 60 duplicate occurrences. Further inspection using keep=False displayed 120 rows, representing 60 duplicate pairs. The duplicate records were removed, leaving 60,000 unique order records.

### 6.9 Invalid Value Analysis

After handling missing values and duplicate records, the numerical fields are checked for invalid or suspicious values.

The validation focuses on identifying values that are inconsistent with the business meaning of each field.

The following fields will be reviewed:

* Food_Amount — should not normally be negative.
* Delivery_Fee — should not be negative.
* Discount — should not be negative.
* Final_Amount — should not be negative.
* Extremely high values will also be reviewed as potential outliers.

#### 6.9.1 Investigating Negative Food Items

In [77]:
print(orders[orders['Food_Amount']<0].count()['Food_Amount'])



35


In [78]:
display(orders[orders['Food_Amount']<0])


,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount
675,ORD0000676,CUST00596,REST0225,2022-01-18,04:49:56,Delivered,-450.77,31.62,45.08,437.31,UPI,-856.46
3532,ORD0003533,CUST01644,REST0121,2025-05-14,14:14:03,Delivered,-322.59,39.12,32.26,329.45,UPI,-612.92
4703,ORD0004704,CUST00490,REST0001,2025-09-04,23:28:44,Delivered,-954.48,37.95,238.62,753.81,Debit Card,-1670.34
4936,ORD0004937,CUST01596,REST0068,2024-01-29,13:51:10,Delivered,-286.78,27.02,43.02,270.78,UPI,-530.54
5945,ORD0005946,CUST00857,REST0238,2022-11-09,23:43:46,Cancelled,-244.01,37.01,0.00,281.02,Cash,-488.02
16551,ORD0016552,CUST01323,REST0210,2023-07-05,23:05:14,Delivered,-378.02,56.02,0.00,434.04,Credit Card,-756.04
17241,ORD0017242,CUST01819,REST0103,2024-05-21,05:42:24,Delivered,-611.23,24.46,61.12,574.57,UPI,-1161.34
17497,ORD0017498,CUST01422,REST0067,2024-01-23,20:37:05,Delivered,-435.52,36.16,108.88,362.80,UPI,-762.16
21159,ORD0021160,CUST00979,REST0250,2022-10-16,21:54:52,Delivered,-377.12,38.73,0.00,415.85,Debit Card,-754.24
23786,ORD0023787,CUST02258,REST0066,2022-12-05,17:03:18,Delivered,-526.34,22.28,26.32,522.30,UPI,-1026.36


In [79]:
orders[orders['Food_Amount'] < 0]['Order_Status'].value_counts()

,count
Order_Status,
Delivered,32
Cancelled,3


In [80]:
orders['Calculated_Final_Amount'] = abs(orders['Food_Amount']) + orders['Delivery_Fee'] - orders['Discount']
display(orders[orders['Food_Amount'] < 0][['Calculated_Final_Amount' , 'Final_Amount']])

,Calculated_Final_Amount,Final_Amount
675,437.31,437.31
3532,329.45,329.45
4703,753.81,753.81
4936,270.78,270.78
5945,281.02,281.02
16551,434.04,434.04
17241,574.57,574.57
17497,362.80,362.80
21159,415.85,415.85
23786,522.30,522.30


#### Observation

35 records were identified with negative Food Amount values. The affected records occur across both Delivered and Cancelled orders, indicating that Order Status does not explain the negative values. After converting the Food Amount to its absolute value, the calculated Final Amount matched the recorded Final Amount for all affected records. This validates that the negative signs are data-entry/clerical errors and that the Food Amount values should be corrected to positive values.

#### 6.9.2 Correcting Negative Food Amounts

In [81]:
orders['Food_Amount'] = abs(orders['Food_Amount'])
print(orders[orders['Food_Amount'] < 0].count()['Food_Amount'])

0


In [82]:
display(orders.head())
display(orders.tail())

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount
0,ORD0000001,CUST00613,REST0160,2024-07-03,16:44:28,Delivered,3227.95,34.74,484.19,2778.50,Debit Card,484.19,2778.50
1,ORD0000002,CUST02175,REST0218,2022-06-12,00:48:39,Cancelled,409.02,16.19,40.90,384.31,Wallet,40.90,384.31
2,ORD0000003,CUST00419,REST0115,2023-06-08,03:06:36,Delivered,654.29,45.23,98.14,601.38,UPI,98.14,601.38
3,ORD0000004,CUST01177,REST0129,2024-02-04,14:05:22,Cancelled,771.58,28.63,0.00,800.21,Debit Card,0.00,800.21
4,ORD0000005,CUST01215,REST0021,2023-04-21,07:27:52,Delivered,195.51,10.00,19.55,185.96,UPI,19.55,185.96


,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount
59995,ORD0059996,CUST02093,REST0138,2025-02-14,21:59:00,Delivered,466.14,33.63,0.00,499.77,Debit Card,0.00,499.77
59996,ORD0059997,CUST01860,REST0030,2022-03-19,21:22:59,Delivered,323.13,26.42,48.47,301.08,UPI,48.47,301.08
59997,ORD0059998,CUST01421,REST0197,2025-05-02,14:29:58,Delivered,183.41,25.87,18.34,190.94,UPI,18.34,190.94
59998,ORD0059999,CUST00661,REST0167,2022-04-27,05:28:39,Delivered,389.92,42.89,38.99,393.82,UPI,38.99,393.82
59999,ORD0060000,CUST02344,REST0224,2022-01-14,07:13:36,Delivered,516.01,35.99,103.20,448.80,Debit Card,103.20,448.80


### 6.10 Delivery Fee Validation

The Delivery_Fee column is checked for negative values because delivery charges should not normally be negative. Any negative values identified will be investigated before deciding whether correction is required.

In [83]:
print(orders[orders['Delivery_Fee'] < 0].count()['Delivery_Fee'])

0


### Observation

No negative values were identified in the Delivery_Fee column. All delivery fee values are non-negative and therefore no correction is required.

### 6.11 Discount Validation

The Discount column was checked for negative values after repairing the missing Discount values.

Initially, 16 negative Discount values were identified. These values were extremely small numbers, such as -1.136868e-13, -2.842171e-14, and -5.684342e-14.

These values were consistent with floating-point precision artifacts rather than genuine negative discounts.

The values were compared with the difference between Food_Amount + Delivery_Fee - Final_Amount, which confirmed that the same insignificant negative values were produced by floating-point calculations.

The affected Discount values were therefore rounded to two decimal places and standardized to 0.00.

The final validation confirmed that 0 negative Discount values remain in the dataset. No genuine negative Discount values were identified.

In [84]:
print(orders[orders['Discount'] < 0].count()['Discount'])

16


In [85]:
display(orders[orders['Discount'] < 0])

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount
1195,ORD0001196,CUST01080,REST0125,2022-12-07,00:12:25,Delivered,513.92,23.02,-1.136868e-13,536.94,Debit Card,-1.136868e-13,536.94
2806,ORD0002807,CUST00223,REST0024,2024-12-14,20:25:40,Delivered,708.17,41.19,-1.136868e-13,749.36,Debit Card,-1.136868e-13,749.36
3756,ORD0003757,CUST00958,REST0070,2023-08-23,12:58:34,Delivered,213.10,42.73,-2.842171e-14,255.83,UPI,-2.842171e-14,255.83
5043,ORD0005044,CUST00976,REST0055,2025-09-04,22:27:29,Delivered,916.06,37.30,-1.136868e-13,953.36,Credit Card,-1.136868e-13,953.36
14744,ORD0014745,CUST02282,REST0244,2023-02-22,12:10:58,Delivered,299.53,29.58,-5.684342e-14,329.11,Credit Card,-5.684342e-14,329.11
20517,ORD0020518,CUST01332,REST0138,2022-03-05,09:13:35,Delivered,448.39,23.90,-5.684342e-14,472.29,UPI,-5.684342e-14,472.29
20968,ORD0020969,CUST00893,REST0216,2022-01-29,21:20:42,Cancelled,402.44,23.97,-5.684342e-14,426.41,Credit Card,-5.684342e-14,426.41
35506,ORD0035507,CUST00110,REST0128,2023-07-31,18:28:02,Delivered,324.84,45.82,-5.684342e-14,370.66,Credit Card,-5.684342e-14,370.66
39659,ORD0039660,CUST01423,REST0109,2023-11-30,18:05:13,Delivered,201.32,47.95,-2.842171e-14,249.27,UPI,-2.842171e-14,249.27
42241,ORD0042242,CUST00321,REST0153,2023-05-03,18:48:52,Delivered,219.39,31.10,-2.842171e-14,250.49,UPI,-2.842171e-14,250.49


In [86]:
orders['neg_discount'] = orders['Food_Amount'] + orders['Delivery_Fee'] - orders['Final_Amount']
display(orders[orders['Discount'] < 0][['Discount' , 'neg_discount']])

,Discount,neg_discount
1195,-1.136868e-13,-1.136868e-13
2806,-1.136868e-13,-1.136868e-13
3756,-2.842171e-14,-2.842171e-14
5043,-1.136868e-13,-1.136868e-13
14744,-5.684342e-14,-5.684342e-14
20517,-5.684342e-14,-5.684342e-14
20968,-5.684342e-14,-5.684342e-14
35506,-5.684342e-14,-5.684342e-14
39659,-2.842171e-14,-2.842171e-14
42241,-2.842171e-14,-2.842171e-14


In [87]:
print(orders[orders['Discount'] < 0].count()['Discount'])

16


### Observation — Invalid Value Analysis

The invalid value analysis identified and investigated potential data-quality issues across the key numerical fields.

Food_Amount: 35 negative values were identified and corrected after validation confirmed that they were sign-related data-entry errors.
Delivery_Fee: No negative values were identified, so no correction was required.
Discount: 16 extremely small negative values were identified as floating-point precision artifacts and standardized to 0.00.
Final_Amount: The relationship between Food_Amount, Delivery_Fee, Discount, and Final_Amount was validated to confirm the corrected values.

After the corrections and final validation, no negative values remain in the validated fields due to the identified data-quality issues.

### 6.12 Outlier Analysis

After correcting invalid values, the numerical fields are reviewed for unusually high or low values that may represent potential outliers.

An outlier is a value that is significantly different from the majority of observations. Outliers do not necessarily indicate errors; they may represent genuine business transactions.

The following numerical fields will be reviewed:
- Food_Amount — unusually high order values
- Delivery_Fee — unusually high delivery charges
- Discount — unusually high discounts
- Final_Amount — unusually high final payable amounts

The analysis will use the Interquartile Range (IQR) method to identify potential outliers.

Potential outliers will be investigated before deciding whether any correction is required.

In [88]:
Q1 = orders['Food_Amount'].quantile(0.25)
Q3 = orders['Food_Amount'].quantile(0.75)

IQR = Q3 - Q1

Lower_Bound = Q1 - 1.5 * IQR
Upper_Bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound Food_Amount is:", round(Lower_Bound,2))
print("Upper Bound Food_Amount is:", round(Upper_Bound,2))

Q1: 290.005
Q3: 608.5425
IQR: 318.5375
Lower Bound Food_Amount is: -187.8
Upper Bound Food_Amount is: 1086.35


In [89]:
outliers_food = orders[
    (orders['Food_Amount'] < Lower_Bound) |
    (orders['Food_Amount'] > Upper_Bound)
]

print("Number of potential Food_Amount outliers:", len(outliers_food))

display(outliers_food)

Number of potential Food_Amount outliers: 2540


,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount,neg_discount
0,ORD0000001,CUST00613,REST0160,2024-07-03,16:44:28,Delivered,3227.95,34.74,484.19,2778.50,Debit Card,4.841900e+02,2778.50,4.841900e+02
6,ORD0000007,CUST01854,REST0009,2025-03-27,19:48:12,Delivered,1247.65,39.42,0.00,1287.07,Debit Card,2.273737e-13,1287.07,2.273737e-13
18,ORD0000019,CUST01799,REST0130,2023-10-27,03:57:51,Delivered,1274.44,47.98,127.44,1194.98,UPI,1.274400e+02,1194.98,1.274400e+02
32,ORD0000033,CUST00403,REST0203,2024-09-28,08:42:26,Delivered,1149.80,47.51,57.49,1139.82,Credit Card,5.749000e+01,1139.82,5.749000e+01
55,ORD0000056,CUST01824,REST0208,2022-02-06,14:07:08,Delivered,1354.29,31.46,0.00,1385.75,Credit Card,0.000000e+00,1385.75,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59909,ORD0059910,CUST02070,REST0014,2025-05-15,03:42:12,Delivered,1193.32,50.68,119.33,1124.67,Credit Card,1.193300e+02,1124.67,1.193300e+02
59911,ORD0059912,CUST01791,REST0006,2025-01-02,16:57:24,Delivered,1728.66,44.05,259.30,1513.41,Credit Card,2.593000e+02,1513.41,2.593000e+02
59915,ORD0059916,CUST00481,REST0001,2023-09-12,00:53:38,Delivered,1178.72,34.76,0.00,1213.48,UPI,0.000000e+00,1213.48,0.000000e+00
59944,ORD0059945,CUST01716,REST0165,2025-05-31,21:19:30,Delivered,1137.04,39.59,113.70,1062.93,Credit Card,1.137000e+02,1062.93,1.137000e+02


In [90]:
print("Highest food amount:", orders['Food_Amount'].max())

Highest food amount: 3500.0


In [91]:
orders[orders['Food_Amount'] == 3500]

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount,neg_discount
7207,ORD0007208,CUST01721,REST0073,2025-07-20,00:30:34,Delivered,3500.0,16.38,0.0,3516.38,Cash,0.0,3516.38,0.0
21614,ORD0021615,CUST01210,REST0003,2022-06-26,22:53:38,Delivered,3500.0,43.83,350.0,3193.83,Debit Card,350.0,3193.83,350.0


In [92]:
orders.sort_values(by = 'Food_Amount' , ascending = False).head(10)

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount,neg_discount
7207,ORD0007208,CUST01721,REST0073,2025-07-20,00:30:34,Delivered,3500.00,16.38,0.00,3516.38,Cash,0.00,3516.38,0.00
21614,ORD0021615,CUST01210,REST0003,2022-06-26,22:53:38,Delivered,3500.00,43.83,350.00,3193.83,Debit Card,350.00,3193.83,350.00
22798,ORD0022799,CUST01875,REST0024,2023-07-05,05:50:19,Delivered,3271.20,34.30,327.12,2978.38,UPI,327.12,2978.38,327.12
0,ORD0000001,CUST00613,REST0160,2024-07-03,16:44:28,Delivered,3227.95,34.74,484.19,2778.50,Debit Card,484.19,2778.50,484.19
11939,ORD0011940,CUST02241,REST0075,2023-07-06,23:37:01,Delivered,3153.07,24.92,315.31,2862.68,Debit Card,315.31,2862.68,315.31
24214,ORD0024215,CUST00840,REST0187,2023-09-22,05:00:12,Delivered,3138.77,33.52,313.88,2858.41,Debit Card,313.88,2858.41,313.88
34273,ORD0034274,CUST01457,REST0067,2023-08-29,17:33:23,Delivered,3085.99,30.93,600.00,2516.92,Wallet,600.00,2516.92,600.00
16973,ORD0016974,CUST01820,REST0206,2024-09-21,20:58:07,Delivered,3075.57,52.00,153.78,2973.79,Credit Card,153.78,2973.79,153.78
55810,ORD0055811,CUST00601,REST0173,2025-03-22,12:50:37,Delivered,2983.07,11.14,596.61,2397.60,Credit Card,596.61,2397.60,596.61
22159,ORD0022160,CUST00533,REST0001,2024-01-22,17:19:13,Delivered,2896.88,31.33,434.53,2493.68,UPI,434.53,2493.68,434.53


In [93]:
check_final_amount = orders.sort_values(by = 'Food_Amount' , ascending = False).head(10)
check_final_amount[['Final_Amount' , 'Calculated_Final_Amount']]

,Final_Amount,Calculated_Final_Amount
7207,3516.38,3516.38
21614,3193.83,3193.83
22798,2978.38,2978.38
0,2778.50,2778.50
11939,2862.68,2862.68
24214,2858.41,2858.41
34273,2516.92,2516.92
16973,2973.79,2973.79
55810,2397.60,2397.60
22159,2493.68,2493.68


In [94]:
print('No. of potential outliers above the Upper Bound:' ,  orders[orders['Food_Amount'] > Upper_Bound].count()['Food_Amount'])
print('No. of potential outliers below the Lower Bound:' ,  orders[orders['Food_Amount'] < Lower_Bound].count()['Food_Amount'])

print(len(orders[orders['Food_Amount'] > Upper_Bound]))

No. of potential outliers above the Upper Bound: 2540
No. of potential outliers below the Lower Bound: 0
2540


### Observation

The IQR method identified 2,540 potential outliers in the Food_Amount column, all of which were above the upper IQR bound of 1,086.35. No records were below the lower bound.

The highest Food_Amount was ₹3,500, and the highest-value transactions were reviewed as representative extreme cases. Their Final_Amount values matched the calculated relationship between Food_Amount, Delivery_Fee, and Discount.

These records were therefore treated as potential statistical outliers rather than confirmed data-quality errors and were retained in the dataset for further analysis.

### 6.13 Outlier Analysis — Delivery Fee

We've completed the outlier analysis for Food_Amount.

Now we follow the same analytical process for Delivery_Fee.

First, what are we trying to find?

We want to identify unusually high or low delivery charges.

We already established earlier that:

Delivery_Fee should not normally be negative.
We found 0 negative values.
Therefore, now we're interested mainly in unusually high delivery fees.
Our approach

We'll again use the IQR method:

- Calculate Q1 — 25th percentile.
- Calculate Q3 — 75th percentile.
- Calculate IQR.
- Calculate the Lower Bound.
- Calculate the Upper Bound.
- Count records outside those boundaries.
- Investigate the potential outliers.
- Decide whether they are genuine transactions or data-quality problems.
- Document the observation.


In [95]:
Q1 = orders['Delivery_Fee'].quantile(.25)
Q3 = orders['Delivery_Fee'].quantile(.75)

IQR = Q3 - Q1

print('Inter Quartile Range is: ' , IQR)

Lower_Bound = Q1 - 1.5  * IQR
Upper_Bound = Q3 + 1.5  * IQR

print('Lower Bound Delivery Fee is: ' , round(Lower_Bound,2))
print('Upper Bound Delivery Fee is: ' , round(Upper_Bound,2))


Inter Quartile Range is:  16.22
Lower Bound Delivery Fee is:  2.52
Upper Bound Delivery Fee is:  67.4


In [96]:
print('No. Of deliverys above Upper Bound: ', orders[orders['Delivery_Fee'] > Upper_Bound].count()['Delivery_Fee'])
print('No. Of deliverys below Lower Bound: ', orders[orders['Delivery_Fee'] < Lower_Bound].count()['Delivery_Fee'])

No. Of deliverys above Upper Bound:  205
No. Of deliverys below Lower Bound:  0


In [97]:
orders[orders['Delivery_Fee'] > Upper_Bound]

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount,neg_discount
223,ORD0000224,CUST00698,REST0205,2022-05-05,01:30:34,Delivered,322.99,68.82,48.45,343.36,Wallet,48.45,343.36,48.45
365,ORD0000366,CUST00206,REST0078,2023-01-17,01:56:28,Delivered,863.63,70.57,86.36,847.84,UPI,86.36,847.84,86.36
420,ORD0000421,CUST00403,REST0225,2024-09-09,11:33:51,Delivered,362.33,68.69,0.00,431.02,Credit Card,0.00,431.02,0.00
847,ORD0000848,CUST00539,REST0191,2022-01-24,09:22:41,Delivered,369.36,80.39,55.40,394.35,Cash,55.40,394.35,55.40
860,ORD0000861,CUST00432,REST0104,2022-02-10,08:13:13,Cancelled,239.80,71.29,11.99,299.10,UPI,11.99,299.10,11.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58734,ORD0058735,CUST02329,REST0124,2024-01-28,06:49:45,Delivered,974.02,69.40,48.70,994.72,Credit Card,48.70,994.72,48.70
59413,ORD0059414,CUST01174,REST0182,2022-04-30,16:00:46,Delivered,390.41,85.63,58.56,417.48,Credit Card,58.56,417.48,58.56
59919,ORD0059920,CUST02200,REST0133,2024-11-29,05:13:18,Delivered,256.19,74.66,25.62,305.23,Wallet,25.62,305.23,25.62
59950,ORD0059951,CUST00924,REST0200,2022-11-14,00:56:18,Delivered,161.17,68.39,40.29,189.27,UPI,40.29,189.27,40.29


In [98]:
print('Maximum delivery_fee among the Upper Bound is :' , orders[orders['Delivery_Fee'] > Upper_Bound].max()['Delivery_Fee'])
print('Minimum delivery_fee among the Upper Bound is :' , orders[orders['Delivery_Fee'] > Upper_Bound].min()['Delivery_Fee'])

Maximum delivery_fee among the Upper Bound is : 85.63
Minimum delivery_fee among the Upper Bound is : 67.43


In [99]:
check_final_amount = orders[orders['Delivery_Fee'] > Upper_Bound]

display(check_final_amount[(check_final_amount['Final_Amount'] - check_final_amount['Calculated_Final_Amount']) != 0][['Final_Amount' , 'Calculated_Final_Amount']])

unmatch = check_final_amount[(check_final_amount['Final_Amount'] - check_final_amount['Calculated_Final_Amount']) != 0][['Final_Amount' , 'Calculated_Final_Amount']]

print('No. of records where Final_Amount and Calculated_Final_Amount do not match is: ' , len(unmatch))

,Final_Amount,Calculated_Final_Amount
2414,491.89,491.89
3168,349.42,349.42
3609,645.18,645.18
3687,224.58,224.58
5421,225.26,225.26
6107,417.85,417.85
6726,510.91,510.91
7995,916.67,916.67
8744,493.27,493.27
10740,233.81,233.81


No. of records where Final_Amount and Calculated_Final_Amount do not match is:  47


In [100]:
display(
    unmatch.assign(
        Diff_Check = unmatch['Final_Amount'] -
                     unmatch['Calculated_Final_Amount']
    )[['Final_Amount', 'Calculated_Final_Amount', 'Diff_Check']]
)

,Final_Amount,Calculated_Final_Amount,Diff_Check
2414,491.89,491.89,-1.136868e-13
3168,349.42,349.42,5.684342e-14
3609,645.18,645.18,-1.136868e-13
3687,224.58,224.58,2.842171e-14
5421,225.26,225.26,-2.842171e-14
6107,417.85,417.85,5.684342e-14
6726,510.91,510.91,5.684342e-14
7995,916.67,916.67,-1.136868e-13
8744,493.27,493.27,-1.136868e-13
10740,233.81,233.81,2.842171e-14


In [101]:
## rounded the difference to 2 decimal places

unmatch = check_final_amount[(check_final_amount['Final_Amount'] - check_final_amount['Calculated_Final_Amount']).round(2) != 0][['Final_Amount' , 'Calculated_Final_Amount']]

print('No. of records where Final_Amount and Calculated_Final_Amount do not match is: ' , len(unmatch))

No. of records where Final_Amount and Calculated_Final_Amount do not match is:  0


### Observation

The IQR method identified 205 potential outliers in Delivery_Fee, all above the upper bound of 67.40. The outlier values ranged from 67.43 to 85.63. These records were investigated by comparing Final_Amount with Calculated_Final_Amount. After accounting for floating-point precision and rounding the difference to two decimal places, all 205 records showed no actual difference. Therefore, these records were treated as genuine high delivery-fee transactions rather than data-quality errors and were retained for further analysis.

### 6.14 Outlier Analysis — Discount

In [102]:
## Calculated the discounted rate and checked if discount rate exceeds 100%

orders['Discount_Rate'] = round(orders['Discount'] / (orders['Food_Amount'] + orders['Delivery_Fee']),2)
print('No. of Discount rate exceeding 100% is:  ' , orders[orders['Discount_Rate']>1]['Discount_Rate'].count())

No. of Discount rate exceeding 100% is:   0


In [103]:
## Calculated the discounted rate and checked if discount rate below 0%

print('No. of Discount rate below 0% (negative) is:  ' , orders[orders['Discount_Rate'] < 0]['Discount_Rate'].count())

No. of Discount rate below 0% (negative) is:   0


In [104]:
Q1 = orders['Discount_Rate'].quantile(.25)
Q3 = orders['Discount_Rate'].quantile(.75)

IQR = Q3 - Q1

print('Inter Quartile Range is: ' , IQR)

Lower_Bound = round((Q1 - 1.5) * IQR , 2)
Upper_Bound = round((Q3 + 1.5) * IQR , 2)

print('Lower Bound Discount Rate is: ' , Lower_Bound)
print('Upper Bound Discount Rate is: ' , Upper_Bound)

Inter Quartile Range is:  0.13
Lower Bound Discount Rate is:  -0.2
Upper Bound Discount Rate is:  0.21


In [105]:
## Calculating the no. of discount above 21%

print('No. of orders with discount rate greater than Upper Bound is: ', orders[orders['Discount_Rate'] > 0.21]['Discount_Rate'].count())

No. of orders with discount rate greater than Upper Bound is:  2023


In [106]:
high_discount = orders[orders['Discount_Rate'] > 0.21]
high_discount.head()

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount,neg_discount,Discount_Rate
16,ORD0000017,CUST00238,REST0214,2022-02-13,00:57:04,Delivered,646.43,32.52,161.61,517.34,Cash,161.61,517.34,161.61,0.24
35,ORD0000036,CUST01972,REST0159,2022-10-11,01:23:30,Delivered,222.29,33.67,55.57,200.39,UPI,55.57,200.39,55.57,0.22
43,ORD0000044,CUST01591,REST0145,2022-04-13,03:29:51,Delivered,473.65,33.05,118.41,388.29,UPI,118.41,388.29,118.41,0.23
99,ORD0000100,CUST02221,REST0159,2025-03-01,21:22:15,Delivered,986.04,17.11,246.51,756.64,Debit Card,246.51,756.64,246.51,0.25
176,ORD0000177,CUST00049,REST0235,2022-06-23,15:21:50,Cancelled,843.35,55.29,210.84,687.80,UPI,210.84,687.80,210.84,0.23


In [107]:
## Compare Final_Amount with Calculated_Final_Amount of the this 2023 orders

display(high_discount.assign(Diff_Check = round(high_discount['Final_Amount'] , 2) - round(high_discount['Calculated_Final_Amount'] , 2))[['Final_Amount', 'Calculated_Final_Amount', 'Diff_Check']])

print('No. of records where Final_Amount and Calculated_Final_Amount match is: ' , len(high_discount))


,Final_Amount,Calculated_Final_Amount,Diff_Check
16,517.34,517.34,0.0
35,200.39,200.39,0.0
43,388.29,388.29,0.0
99,756.64,756.64,0.0
176,687.80,687.80,0.0
...,...,...,...
59827,478.41,478.41,0.0
59859,360.68,360.68,0.0
59870,656.01,656.01,0.0
59893,569.89,569.89,0.0


No. of records where Final_Amount and Calculated_Final_Amount match is:  2023


### Observation

The IQR method identified 2,023 potential high-discount outliers. These records were investigated by comparing the recorded Final_Amount with the calculated final amount. All 2,023 records matched, indicating that the high discounts are internally consistent with the transaction calculations. Therefore, they are treated as genuine high-discount transactions rather than data-quality errors and are retained for further analysis.

### 6.15 Final Amount Validation

Final_Amount is a derived field calculated using the underlying transaction components:

Final Amount = Food Amount + Delivery Fee - Discount

Unlike Food_Amount, Delivery_Fee, and Discount, Final_Amount does not need a separate outlier analysis simply because it is a numerical field. An unusually high final amount can be a legitimate result of a high-value order.

Therefore, the validation focuses on reconciling the recorded Final_Amount with the calculated value.

Validation Approach

For every transaction:

- Calculate the expected Final Amount using Food_Amount + Delivery_Fee - Discount.
- Compare the calculated value with the recorded Final_Amount.
- Identify any records where the two values do not match.
- Investigate any mismatches found.
- Document the final observation.

Objective

The objective is to confirm that the Final_Amount field is mathematically consistent with its underlying transaction components and does not contain calculation or data-entry errors.

Expected Outcome

If no mismatches are identified, Final_Amount will be considered validated and consistent with the underlying transaction components.

In [108]:
## Checking Final_Amount with Calculated_Final_Amount

orders['Diff_Check'] = round(orders['Final_Amount'] , 2) - round(orders['Calculated_Final_Amount'] , 2)
print('The no of mismatch between Final_Amount and Calculated_Final_Amount is: ' ,(orders[orders['Diff_Check'] != 0]).count()['Diff_Check'])

The no of mismatch between Final_Amount and Calculated_Final_Amount is:  0


### Observation

The Final_Amount field was validated against the corresponding Calculated_Final_Amount for all transactions.

The comparison identified 0 mismatches, confirming that the recorded Final_Amount values are mathematically consistent with the underlying Food_Amount, Delivery_Fee, and Discount components.

Therefore, no correction was required for the Final_Amount field, and the column was retained for further analysis.

## 7. Customers Table - Data Cleaning and Validation

### Objective

The Customers table contains customer-level information used to understand
customer demographics, location, and customer behavior.

The objective of this step is to assess the quality of the Customers table,
identify missing, duplicate, inconsistent, or invalid records, and correct
issues only where the correction can be logically justified.

### Validation Approach

The Customers table will be evaluated for:

- Dataset structure and dimensions
- Data types
- Missing values
- Duplicate records
- Invalid or inconsistent values
- Unusual values requiring further investigation
- Logical relationships between relevant fields

Each identified issue will be investigated before making any correction.
Unusual values will not automatically be treated as errors.

In [109]:
## Loading/reading of the customer table

customers = pd.read_csv('/content/customers.csv')
customers.head()

,Customer_ID,Customer_Name,Age,Gender,City,Customer_Segment,Signup_Date
0,CUST00001,Customer 00001,21,Female,Mumbai,Regular,2024-09-10
1,CUST00002,Customer 00002,24,Female,Jaipur,Regular,2024-05-17
2,CUST00003,Customer 00003,52,Male,Pune,Regular,2023-07-16
3,CUST00004,Customer 00004,26,Male,Hyderabad,Regular,2022-07-31
4,CUST00005,Customer 00005,25,Male,Delhi,Regular,2025-10-14


In [110]:
## Checking the Customers table structure, dimensions, null values and data types

customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Customer_ID       2500 non-null   object
 1   Customer_Name     2500 non-null   object
 2   Age               2500 non-null   int64 
 3   Gender            2500 non-null   object
 4   City              2500 non-null   object
 5   Customer_Segment  2500 non-null   object
 6   Signup_Date       2500 non-null   object
dtypes: int64(1), object(6)
memory usage: 136.8+ KB


In [111]:
## Checking the statistical summary and valid range of the Age column

customers.describe()

,Age
count,2500.000000
mean,39.039600
std,12.684321
min,12.000000
25%,28.750000
50%,39.000000
75%,49.000000
max,85.000000


In [112]:
## Inspecting the Signup_Date column to verify the stored date values

customers['Signup_Date']


,Signup_Date
0,2024-09-10
1,2024-05-17
2,2023-07-16
3,2022-07-31
4,2025-10-14
...,...
2495,2023-09-14
2496,2022-08-04
2497,2023-12-18
2498,2024-02-04


In [113]:
## Checking the minimum and maximum Signup_Date to identify any unusual or invalid dates

print(customers['Signup_Date'].min())
print(customers['Signup_Date'].max())

2022-01-01
2025-12-30


In [114]:
## Checking for completely duplicated records and duplicate Customer_ID values

print('No. of Duplicate rows is: ' , customers.duplicated().sum())
print('No. of Duplicate Customer_ID is: ' , customers['Customer_ID'].duplicated().sum())


No. of Duplicate rows is:  0
No. of Duplicate Customer_ID is:  0


In [115]:
## Inspecting a sample record to verify the Customers table structure and field values

customers.head(1)

,Customer_ID,Customer_Name,Age,Gender,City,Customer_Segment,Signup_Date
0,CUST00001,Customer 00001,21,Female,Mumbai,Regular,2024-09-10


In [116]:
## Checking unique values in categorical columns to identify invalid or inconsistent values

print(customers['City'].unique())
print(customers['Gender'].unique())
print(customers['Customer_Segment'].unique())

['Mumbai' 'Jaipur' 'Pune' 'Hyderabad' 'Delhi' 'Kolkata' 'Chennai'
 'Bengaluru']
['Female' 'Male' 'Other']
['Regular' 'Premium' 'New']


In [117]:
## Validating that every Customer_ID in the Orders table has a corresponding record in the Customers table

print('How many customer_ID under orders table is available in customer_ID under customers table: ', orders['Customer_ID'].isin(customers['Customer_ID']).sum())

print('How many customer_ID under orders table is not available in customer_ID under customers table: ', (~orders['Customer_ID'].isin(customers['Customer_ID'])).sum())


How many customer_ID under orders table is available in customer_ID under customers table:  60000
How many customer_ID under orders table is not available in customer_ID under customers table:  0


In [118]:
## Validating the one-to-one relationship between Customer_ID and Customer_Name

customer_name_check = customers.groupby('Customer_ID')['Customer_Name'].nunique()
print(customer_name_check[customer_name_check > 1].count())

0


In [119]:
## Converting the Signup_Date column from object dtype to datetime dtype for accurate date-based analysis

customers['Signup_Date'] = pd.to_datetime(customers['Signup_Date'])
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Customer_ID       2500 non-null   object        
 1   Customer_Name     2500 non-null   object        
 2   Age               2500 non-null   int64         
 3   Gender            2500 non-null   object        
 4   City              2500 non-null   object        
 5   Customer_Segment  2500 non-null   object        
 6   Signup_Date       2500 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(5)
memory usage: 136.8+ KB


## Conclusion

- The Customers table contains **2,500 records and 7 columns**, with no missing/null values identified.
- `Customer_ID` contains **no duplicate values**, and no completely duplicated records were found.
- `Age` is stored as an integer and ranges from **12 to 85**. No immediately invalid age values were identified.
- `City`, `Gender`, and `Customer_Segment` contain valid and consistent categorical values with no unexpected categories identified.
- `Signup_Date` values range from **2022-01-01 to 2025-12-30** and were found to be valid date values. The column was initially stored as `object` and was converted to **datetime** for accurate date-based analysis.
- The **Customer_ID → Customer_Name** relationship was validated, with each Customer_ID mapping to a single Customer_Name.
- Referential integrity with the Orders table was validated. All **60,000 Customer_ID references** in Orders have a corresponding Customer_ID in the Customers table.
- No data correction was required for missing, duplicate, categorical, or logical relationship issues.



The Customers table was found to be **complete, consistent, and logically valid** after validation. The only data-quality correction required was converting `Signup_Date` from `object` to `datetime`. The table is now ready for further analysis.

## 8. Restaurants Table - Data Cleaning and Validation

### Objective
The Restaurants table contains restaurant-level information used to understand restaurant identity, location, cuisine, ratings, pricing, and operational characteristics.
The objective of this step is to assess the quality of the Restaurants table, identify missing, duplicate, inconsistent, or invalid records, and correct issues only where the correction can be logically justified.


### Validation Approach
The Restaurants table will be evaluated for:
- Dataset structure and dimensions
- Data types
- Missing values
- Duplicate records
- Duplicate Restaurant IDs
- Invalid or inconsistent values
- Unusual values requiring further investigation
- Validity of categorical and numerical fields
- Logical relationships between relevant fields

Each identified issue will be investigated before making any correction. Unusual values will not automatically be treated as errors.

In [120]:
## Loading/reading of the Restaurants table

restaurants = pd.read_csv('/content/restaurants.csv')
restaurants.head()

,Restaurant_ID,Restaurant_Name,Cuisine,City,Restaurant_Rating,Avg_Preparation_Min
0,REST0001,Urban Kitchen 001,Biryani,Mumbai,4.4,30
1,REST0002,Street Bites 002,Desserts,Delhi,3.9,25
2,REST0003,Fresh Bites 003,Fast Food,Mumbai,3.9,31
3,REST0004,Urban Express 004,Healthy,Jaipur,3.9,17
4,REST0005,Urban Hub 005,South Indian,Chennai,4.8,40


In [121]:
## Checking the Restaurants table structure, dimensions, null values and data types

restaurants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Restaurant_ID        250 non-null    object 
 1   Restaurant_Name      250 non-null    object 
 2   Cuisine              250 non-null    object 
 3   City                 250 non-null    object 
 4   Restaurant_Rating    238 non-null    float64
 5   Avg_Preparation_Min  250 non-null    int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 11.8+ KB


### Observations

- Dataset contains 250 rows and 6 columns.
- Restaurant_ID, Restaurant_Name, Cuisine, City, and Avg_Preparation_Min contain 250 non-null values.
- Restaurant_Rating contains 238 non-null values, indicating 12 missing values.
- Data types are otherwise appropriate: rating is numeric (float64) and preparation time is integer (int64).
- The missing ratings require investigation before deciding whether they should be corrected or retained.

In [122]:
## Checking the missing values in Restaurant_Rating

restaurants[restaurants['Restaurant_Rating'].isnull()]

,Restaurant_ID,Restaurant_Name,Cuisine,City,Restaurant_Rating,Avg_Preparation_Min
24,REST0025,Street Express 025,South Indian,Mumbai,NaN,34
37,REST0038,Royal Hub 038,Italian,Mumbai,NaN,13
61,REST0062,Urban House 062,Healthy,Jaipur,NaN,28
71,REST0072,Spice Express 072,South Indian,Pune,NaN,40
103,REST0104,Fresh Cafe 104,South Indian,Hyderabad,NaN,27
115,REST0116,Food Express 116,North Indian,Jaipur,NaN,38
128,REST0129,Kitchen Bites 129,Desserts,Jaipur,NaN,30
214,REST0215,Taste Kitchen 215,Chinese,Mumbai,NaN,36
217,REST0218,Street Kitchen 218,Chinese,Bengaluru,NaN,19
237,REST0238,Food Express 238,Biryani,Kolkata,NaN,25


### Observation
- 12 restaurants have missing Restaurant_Rating values.
- The missing ratings cannot be logically or reliably inferred from the available fields.
- No valid business rule was identified to determine the original ratings.
- The affected records contain other valid restaurant information and should not be removed.
- The missing ratings will therefore be retained as NaN rather than introducing assumed or fabricated values.

In [123]:
## Checking the statistical summary and valid range of numerical columns

restaurants.describe()

,Restaurant_Rating,Avg_Preparation_Min
count,238.000000,250.000000
mean,4.043697,25.776000
std,0.453796,8.300447
min,2.700000,12.000000
25%,3.800000,19.000000
50%,4.100000,26.000000
75%,4.400000,32.000000
max,5.000000,40.000000


### Observation
- Restaurant ratings range from 2.7 to 5.0 and fall within the expected rating scale.
- Average preparation time ranges from 12 to 40 minutes, with no negative, zero, or obviously unrealistic values.
- No numerical anomalies requiring correction were identified.

In [124]:
## Checking unique values and category counts in Cuisine and City to identify invalid or inconsistent values

print('different Cuisine categories are: ', restaurants['Cuisine'].unique())
print()
print('Number of different Cuisine categories are: ', restaurants['Cuisine'].nunique())
print()
print('different City categories are: ', restaurants['City'].unique())
print()
print('Number of different City categories are: ', restaurants['City'].nunique())

different Cuisine categories are:  ['Biryani' 'Desserts' 'Fast Food' 'Healthy' 'South Indian' 'North Indian'
 'Chinese' 'Italian']

Number of different Cuisine categories are:  8

different City categories are:  ['Mumbai' 'Delhi' 'Jaipur' 'Chennai' 'Hyderabad' 'Kolkata' 'Pune'
 'Bengaluru']

Number of different City categories are:  8


### Observation
- Cuisine contains 8 unique values, all of which are reasonable and consistently formatted.
- City contains 8 unique values, all of which are valid and consistently formatted.
- No invalid, misspelled, or inconsistent categorical values were identified.
- No correction was required for these categorical fields.

In [125]:
## Checking Restaurant_ID uniqueness to ensure each restaurant has a unique identifier

check_restaurant_ID = restaurants.groupby('Restaurant_ID')['Restaurant_Name'].count()
check_restaurant_ID[restaurants.groupby('Restaurant_ID')['Restaurant_Name'].count() > 1 ]


,Restaurant_Name
Restaurant_ID,


### Observation
- Each Restaurant_ID is associated with only one Restaurant_Name.
- No duplicate or conflicting Restaurant_ID-to-Restaurant_Name relationships were identified.
- Therefore, Restaurant_ID can be considered a reliable unique identifier for the Restaurants table.

In [126]:
## Checking for duplicate Restaurant_ID records to ensure each restaurant appears only once
print('No. of duplicate value under ID columns is: ' , restaurants['Restaurant_ID'].duplicated().sum())

No. of duplicate value under ID columns is:  0


### Observation
- No duplicate Restaurant_ID values were identified.
- Each restaurant appears only once in the table.
- No duplicate records were identified based on Restaurant_ID.
- Therefore, no records were removed from the Restaurants table.

## Conclusion

- The Restaurants table contains 250 records and 6 columns, with appropriate data types for the respective fields.
- 12 missing values were identified in `Restaurant_Rating`. These values could not be logically or reliably inferred from the available data, so the records were retained without modification.
- `Restaurant_Rating` values range from 2.7 to 5.0, which falls within the expected rating scale.
- `Avg_Preparation_Min` ranges from 12 to 40 minutes, with no negative, zero, or obviously unrealistic values identified.
- All observed Cuisine and City values are valid and consistently formatted, with no apparent spelling or category inconsistencies.
- `Restaurant_ID` values are unique, and the relationship between `Restaurant_ID` and `Restaurant_Name` is consistent.
- No data-quality issues requiring correction were identified beyond the missing ratings, which were intentionally retained.
- The Restaurants table is therefore considered cleaned and validated and is ready for further analysis.

# 9. Delivery Partners Table - Data Cleaning and Validation

## Objective

The Delivery Partners table contains delivery-partner-level information used to understand delivery partner identity, availability, location, performance, and operational characteristics.

The objective of this step is to assess the quality of the Delivery Partners table, identify missing, duplicate, inconsistent, or invalid records, and correct issues only where the correction can be logically justified.

## Validation Approach

The Delivery Partners table will be evaluated for:

- Dataset structure and dimensions
- Data types
- Missing values
- Duplicate records
- Duplicate Delivery Partner IDs
- Invalid or inconsistent values
- Unusual values requiring further investigation
- Validity of categorical and numerical fields
- Logical relationships between relevant fields

Each identified issue will be investigated before making any correction. Unusual values will not automatically be treated as errors.

In [127]:
## Loading/reading of the Delivery Partners table

delivery_partners = pd.read_csv('/content/delivery_partners.csv')
delivery_partners.head()

,Delivery_Partner_ID,Partner_Name,Vehicle_Type,City,Partner_Rating,Experience_Months
0,DP0001,Partner 0001,Bike,Bengaluru,4.7,13
1,DP0002,Partner 0002,Scooter,Hyderabad,4.5,24
2,DP0003,Partner 0003,Bike,Chennai,3.8,48
3,DP0004,Partner 0004,Cycle,Bengaluru,4.6,38
4,DP0005,Partner 0005,Bike,Chennai,4.6,58


In [128]:
## Checking the Delivery Partners table structure, dimensions, null values and data types

delivery_partners.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Delivery_Partner_ID  700 non-null    object 
 1   Partner_Name         700 non-null    object 
 2   Vehicle_Type         700 non-null    object 
 3   City                 700 non-null    object 
 4   Partner_Rating       700 non-null    float64
 5   Experience_Months    700 non-null    int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 32.9+ KB


### Observation

- The Delivery Partners table contains 700 rows and 6 columns.
- All six columns contain 700 non-null values, indicating that there are no missing values.
- Delivery_Partner_ID, Partner_Name, Vehicle_Type and City are stored as object data types.
- Partner_Rating is stored as a float, which is appropriate for rating values.
- Experience_Months is stored as an integer, which is appropriate for representing completed months of experience.
- No data type or missing-value issues were identified at this stage.

In [129]:
## Checking the statistical summary and valid range of numerical columns

delivery_partners.describe()

,Partner_Rating,Experience_Months
count,700.000000,700.000000
mean,4.194143,30.440000
std,0.347000,17.313341
min,3.300000,1.000000
25%,3.975000,15.750000
50%,4.200000,30.000000
75%,4.400000,45.250000
max,5.000000,60.000000


### Observation

- Partner ratings range from 3.3 to 5.0, which falls within the expected rating scale.
- Experience_Months ranges from 1 to 60 months, with no negative or zero values.
- Both numerical columns contain 700 non-null values.
- No unusual or logically invalid numerical values were identified.
- No correction was required for the numerical fields.

In [130]:
## Checking duplicate Delivery_Partner_ID values to ensure each partner has a unique identifier

print('No. of duplicate ID is: ' ,  delivery_partners['Delivery_Partner_ID'].duplicated().sum())

No. of duplicate ID is:  0


### Observation

- No duplicate Delivery_Partner_ID values were identified.
- Each delivery partner has a unique identifier.
- Therefore, Delivery_Partner_ID can be treated as a reliable unique identifier for the Delivery Partners table.

In [131]:
## Checking unique values in categorical columns to identify invalid or inconsistent values

print('different Vehicle Type categories are: ' , delivery_partners['Vehicle_Type'].unique())
print()
print('Number of Vehicle Type categories are: ' , delivery_partners['Vehicle_Type'].nunique())
print()
print('different Cities categories are: ' , delivery_partners['City'].unique())
print()
print('Number of Cities categories are: ' , delivery_partners['City'].nunique())

different Vehicle Type categories are:  ['Bike' 'Scooter' 'Cycle' 'EV Bike']

Number of Vehicle Type categories are:  4

different Cities categories are:  ['Bengaluru' 'Hyderabad' 'Chennai' 'Pune' 'Delhi' 'Kolkata' 'Jaipur'
 'Mumbai']

Number of Cities categories are:  8


### Observation

- Vehicle_Type contains 4 unique values: Bike, Scooter, Cycle, and EV Bike.
- City contains 8 unique values, all of which are valid and consistently formatted.
- No invalid, misspelled, or inconsistent categorical values were identified.
- No correction was required for these categorical fields.

In [132]:
## Validating the logical relationship between Delivery_Partner_ID and Partner_Name

check_delivery_ID = delivery_partners.groupby('Delivery_Partner_ID')['Partner_Name'].nunique()
check_delivery_ID[check_delivery_ID > 1]


,Partner_Name
Delivery_Partner_ID,


### Observation

- Each Delivery_Partner_ID is associated with only one Partner_Name.
- No duplicate or conflicting Delivery_Partner_ID-to-Partner_Name relationships were identified.
- Therefore, Delivery_Partner_ID can be considered a reliable unique identifier for the Delivery Partners table.

## Conclusion

- The Delivery Partners table contains 700 records and 6 columns with no missing values.
- All columns have appropriate data types for their respective fields.
- Partner ratings range from 3.3 to 5.0, and experience ranges from 1 to 60 months, with no invalid or unusual values identified.
- No duplicate Delivery_Partner_ID values were found.
- Vehicle types and cities contain valid and consistently formatted categorical values.
- Each Delivery_Partner_ID is associated with a single Partner_Name, confirming the expected logical relationship.
- No data quality issues requiring correction were identified.
- Therefore, the Delivery Partners table is considered clean and ready for further analysis.

# 10. Deliveries Table - Data Cleaning and Validation

## Objective

The Deliveries table contains delivery-level information used to understand delivery performance, delivery partners, delivery time, distance, delivery status, and related operational characteristics. The objective of this step is to assess the quality of the Deliveries table, identify missing, duplicate, inconsistent, or invalid records, and correct issues only where the correction can be logically justified.

## Validation Approach
The Deliveries table will be evaluated for:
- Dataset structure and dimensions
- Data types
- Missing values
- Duplicate records
- Duplicate Delivery IDs
- Invalid or inconsistent values
- Unusual values requiring further investigation
- Validity of categorical and numerical fields
- Logical relationships between relevant fields
- Relationships with related tables such as Orders and Delivery Partners

Each identified issue will be investigated before making any correction. Unusual values will not automatically be treated as errors.

In [133]:
## Loading/reading of the Deliveries table

deliveries = pd.read_csv('/content/deliveries.csv')
deliveries.head()

,Delivery_ID,Order_ID,Delivery_Partner_ID,Distance_KM,Estimated_Delivery_Min,Actual_Delivery_Min,Delivery_Status
0,DEL0000001,ORD0000001,DP0260,0.50,58,58,On Time
1,DEL0000002,ORD0000003,DP0661,10.17,52,91,Delayed
2,DEL0000003,ORD0000005,DP0310,10.52,38,69,Delayed
3,DEL0000004,ORD0000006,DP0347,3.71,33,42,Delayed
4,DEL0000005,ORD0000007,DP0483,4.61,51,52,Delayed


In [134]:
## Checking the Deliveries table structure, dimensions, null values and data types

deliveries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55450 entries, 0 to 55449
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Delivery_ID             55450 non-null  object 
 1   Order_ID                55450 non-null  object 
 2   Delivery_Partner_ID     55450 non-null  object 
 3   Distance_KM             55450 non-null  float64
 4   Estimated_Delivery_Min  55450 non-null  int64  
 5   Actual_Delivery_Min     55450 non-null  int64  
 6   Delivery_Status         55450 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 3.0+ MB


In [135]:
## Checking the statistical summary and valid range of numerical columns

deliveries.describe()

,Distance_KM,Estimated_Delivery_Min,Actual_Delivery_Min
count,55450.000000,55450.000000,55450.000000
mean,5.512613,42.390550,58.051253
std,3.711410,10.409717,15.883228
min,0.000000,25.000000,2.000000
25%,2.770000,33.000000,47.000000
50%,4.700000,42.000000,58.000000
75%,7.370000,51.000000,69.000000
max,25.000000,60.000000,102.000000


### Observation

- The Deliveries table contains 55,450 records and 7 columns.
- All columns contain 55,450 non-null values, indicating that no missing values were identified.
- The data types are appropriate for the respective fields, with numerical fields stored as numeric data types.
- Estimated delivery time ranges from 25 to 60 minutes, which appears reasonable.
- Distance ranges from 0 to 25 KM. The minimum value of 0 KM requires further investigation to determine whether it represents a valid scenario or a data issue.
- Actual delivery time ranges from 2 to 102 minutes. The unusually low minimum value of 2 minutes requires further investigation before determining whether it is valid.
- No corrections will be made at this stage; unusual values will be investigated before making any data-cleaning decisions.

In [136]:
## Investigating zero values in Distance_KM

deliveries[deliveries['Distance_KM'] == 0]

,Delivery_ID,Order_ID,Delivery_Partner_ID,Distance_KM,Estimated_Delivery_Min,Actual_Delivery_Min,Delivery_Status
2802,DEL0002803,ORD0003026,DP0275,0.0,31,63,Delayed
4287,DEL0004288,ORD0004637,DP0216,0.0,46,63,Delayed
4954,DEL0004955,ORD0005346,DP0382,0.0,39,62,Delayed
6696,DEL0006697,ORD0007232,DP0510,0.0,59,79,Delayed
9268,DEL0009269,ORD0010019,DP0519,0.0,27,62,Delayed
18322,DEL0018323,ORD0019877,DP0319,0.0,60,86,Delayed
21626,DEL0021627,ORD0023444,DP0399,0.0,38,73,Delayed
25257,DEL0025258,ORD0027371,DP0325,0.0,31,64,Delayed
25640,DEL0025641,ORD0027783,DP0474,0.0,28,23,On Time
26359,DEL0026360,ORD0028560,DP0312,0.0,53,88,Delayed


In [137]:
## Investigating unusually low values in Actual_Delivery_Min

deliveries[deliveries['Actual_Delivery_Min'] == 2]

,Delivery_ID,Order_ID,Delivery_Partner_ID,Distance_KM,Estimated_Delivery_Min,Actual_Delivery_Min,Delivery_Status
3541,DEL0003542,ORD0003825,DP0302,7.36,53,2,Delayed
4241,DEL0004242,ORD0004587,DP0071,10.58,59,2,Delayed
5910,DEL0005911,ORD0006391,DP0045,2.19,35,2,On Time
6602,DEL0006603,ORD0007128,DP0492,17.96,59,2,Delayed
6622,DEL0006623,ORD0007150,DP0029,2.46,25,2,Delayed
11022,DEL0011023,ORD0011932,DP0454,1.46,45,2,Delayed
20766,DEL0020767,ORD0022520,DP0037,2.60,45,2,Delayed
21881,DEL0021882,ORD0023720,DP0686,12.01,54,2,Delayed
22716,DEL0022717,ORD0024636,DP0307,13.62,52,2,Delayed
28022,DEL0028023,ORD0030353,DP0073,7.19,47,2,Delayed


### Observation

- Multiple records contain a Distance_KM value of 0. These records have valid delivery, order, partner, and time information, so the zero values cannot be conclusively identified as errors from the Deliveries table alone.
- Multiple records have an Actual_Delivery_Min value of 2 minutes, including deliveries with considerably higher distances and estimated delivery times.
- The 2-minute actual delivery time appears unusually low and requires further validation before determining whether it represents a data-quality issue.
- No values were corrected at this stage because the available evidence is insufficient to justify a correction.

### Observation

- Delivery_Status does not consistently align with the relationship between Estimated_Delivery_Min and Actual_Delivery_Min.
- Several records show an Actual_Delivery_Min lower than the estimated time but are still marked as Delayed.
- Therefore, Delivery_Status cannot be reliably derived from the two delivery-time fields using a simple comparison rule.
- No correction will be made to Delivery_Status without additional evidence regarding the business rule used to determine the status.

In [138]:
## Checking Delivery_ID uniqueness to ensure each delivery has a unique identifier

deliveries['Delivery_ID'].nunique()

55450

### Observation
- The Deliveries table contains 55,450 records.
- The number of unique Delivery_ID values is also 55,450.
- Therefore, every delivery record has a unique Delivery_ID.
- Delivery_ID can be treated as a reliable unique identifier for the Deliveries table.

In [139]:
## Checking for duplicate Delivery_ID records
print(deliveries['Delivery_ID'].duplicated().sum())

0


### Observation
Delivery_ID Uniqueness
- The Deliveries table contains 55,450 records.
- The number of unique Delivery_ID values is also 55,450.
- No duplicate Delivery_ID values were identified.
- Therefore, Delivery_ID can be treated as a reliable unique identifier for the Deliveries table.

In [140]:
## Checking the logical relationship between Delivery_ID and Order_ID

check_deliveries = deliveries.groupby('Delivery_ID')['Order_ID'].nunique()
check_deliveries[check_deliveries > 1]

,Order_ID
Delivery_ID,


### Observation
Delivery_ID and order ID relationship
Delivery_ID and Order_ID Relationship
- Each Delivery_ID is associated with only one Order_ID.
- No Delivery_ID was found to be linked to multiple Order_ID values.
- Therefore, the relationship between Delivery_ID and Order_ID is consistent within the Deliveries table.
- No correction is required.

In [141]:
## Validating Order_ID references against the Orders table

print(deliveries['Order_ID'].isin(orders['Order_ID']).sum())

55450


### Observation

- All 55,450 Order_ID values in the Deliveries table have matching records in the Orders table.
- No unmatched Order_ID references were identified.
- Therefore, the Order_ID references in the Deliveries table are valid.
- No correction was required for this relationship.

In [149]:
## Validating Delivery_Partner_ID references against the Delivery Partners table
##deliveries.head()

print(deliveries['Delivery_Partner_ID'].isin(delivery_partners['Delivery_Partner_ID']).sum())


55450


### Observation

- All 55,450 Delivery_Partner_ID values in the Deliveries table have matching records in the Delivery Partners table.
- No unmatched Delivery_Partner_ID references were identified.
- Therefore, the Delivery_Partner_ID references in the Deliveries table are valid.
- No correction was required for this relationship.

In [151]:
## Checking unique values in Delivery_Status to identify invalid or inconsistent categories
##deliveries.head()
print('The different categories are: ' , deliveries['Delivery_Status'].unique())
print()
print('The no. of different categories are: ' , deliveries['Delivery_Status'].nunique())



The different categories are:  ['On Time' 'Delayed']

The no. of different categories are:  2


### Observation

- Delivery_Status contains two unique categories: On Time and Delayed.
- Both categories are meaningful for classifying delivery outcomes.
- No invalid, misspelled, or inconsistently formatted status values were identified.
- No correction was required for the categorical values.

In [158]:
## Investigating inconsistencies between Estimated_Delivery_Min, Actual_Delivery_Min and Delivery_Status
##deliveries.head()

display(deliveries[(deliveries['Estimated_Delivery_Min'] <= deliveries['Actual_Delivery_Min']) & (deliveries['Delivery_Status'] == 'On Time')])
display(deliveries[(deliveries['Estimated_Delivery_Min'] > deliveries['Actual_Delivery_Min']) & (deliveries['Delivery_Status'] == 'Delayed')])

,Delivery_ID,Order_ID,Delivery_Partner_ID,Distance_KM,Estimated_Delivery_Min,Actual_Delivery_Min,Delivery_Status
0,DEL0000001,ORD0000001,DP0260,0.50,58,58,On Time
68,DEL0000069,ORD0000079,DP0149,5.28,37,37,On Time
134,DEL0000135,ORD0000149,DP0681,12.69,27,27,On Time
156,DEL0000157,ORD0000176,DP0062,4.71,29,29,On Time
157,DEL0000158,ORD0000178,DP0179,5.75,58,58,On Time
...,...,...,...,...,...,...,...
55266,DEL0055267,ORD0059810,DP0447,4.32,27,27,On Time
55268,DEL0055269,ORD0059812,DP0510,2.97,37,37,On Time
55320,DEL0055321,ORD0059867,DP0426,3.14,51,51,On Time
55384,DEL0055385,ORD0059932,DP0694,2.62,36,36,On Time


,Delivery_ID,Order_ID,Delivery_Partner_ID,Distance_KM,Estimated_Delivery_Min,Actual_Delivery_Min,Delivery_Status
3541,DEL0003542,ORD0003825,DP0302,7.36,53,2,Delayed
4241,DEL0004242,ORD0004587,DP0071,10.58,59,2,Delayed
6602,DEL0006603,ORD0007128,DP0492,17.96,59,2,Delayed
6622,DEL0006623,ORD0007150,DP0029,2.46,25,2,Delayed
11022,DEL0011023,ORD0011932,DP0454,1.46,45,2,Delayed
20766,DEL0020767,ORD0022520,DP0037,2.60,45,2,Delayed
21881,DEL0021882,ORD0023720,DP0686,12.01,54,2,Delayed
22716,DEL0022717,ORD0024636,DP0307,13.62,52,2,Delayed
28022,DEL0028023,ORD0030353,DP0073,7.19,47,2,Delayed
29268,DEL0029269,ORD0031704,DP0162,4.80,58,2,Delayed


### Observation

- Delivery_Status contains two categories: On Time and Delayed.
- The initial validation identified records where the status agrees with the comparison between estimated and actual delivery times.
- Several records contain unusually low actual delivery times, including values of 2 minutes, despite considerably higher estimated delivery times and delivery distances.
- Further validation is required to quantify records where Delivery_Status contradicts the expected time-based classification.
- No changes were made to Delivery_Status because the underlying business rule has not yet been independently confirmed.

In [159]:
## Identifying records where Delivery_Status contradicts the expected delivery-time classification

display(deliveries[(deliveries['Estimated_Delivery_Min'] <= deliveries['Actual_Delivery_Min']) & (deliveries['Delivery_Status'] == 'Delayed')])
display(deliveries[(deliveries['Estimated_Delivery_Min'] > deliveries['Actual_Delivery_Min']) & (deliveries['Delivery_Status'] == 'On Time')])

,Delivery_ID,Order_ID,Delivery_Partner_ID,Distance_KM,Estimated_Delivery_Min,Actual_Delivery_Min,Delivery_Status
1,DEL0000002,ORD0000003,DP0661,10.17,52,91,Delayed
2,DEL0000003,ORD0000005,DP0310,10.52,38,69,Delayed
3,DEL0000004,ORD0000006,DP0347,3.71,33,42,Delayed
4,DEL0000005,ORD0000007,DP0483,4.61,51,52,Delayed
5,DEL0000006,ORD0000009,DP0262,6.97,39,56,Delayed
...,...,...,...,...,...,...,...
55444,DEL0055445,ORD0059995,DP0475,2.43,57,72,Delayed
55445,DEL0055446,ORD0059996,DP0674,7.79,59,60,Delayed
55446,DEL0055447,ORD0059997,DP0628,6.08,60,76,Delayed
55447,DEL0055448,ORD0059998,DP0457,1.79,36,51,Delayed


,Delivery_ID,Order_ID,Delivery_Partner_ID,Distance_KM,Estimated_Delivery_Min,Actual_Delivery_Min,Delivery_Status
6,DEL0000007,ORD0000010,DP0138,4.40,28,23,On Time
9,DEL0000010,ORD0000013,DP0457,3.38,38,37,On Time
19,DEL0000020,ORD0000023,DP0448,9.73,26,24,On Time
31,DEL0000032,ORD0000037,DP0274,1.22,40,35,On Time
37,DEL0000038,ORD0000043,DP0580,3.18,31,26,On Time
...,...,...,...,...,...,...,...
55415,DEL0055416,ORD0059966,DP0119,1.87,35,34,On Time
55432,DEL0055433,ORD0059983,DP0038,4.69,48,47,On Time
55434,DEL0055435,ORD0059985,DP0139,4.74,50,46,On Time
55441,DEL0055442,ORD0059992,DP0086,9.65,46,42,On Time


### Observation

- Delivery_Status was compared against the expected relationship between Estimated_Delivery_Min and Actual_Delivery_Min.
- A total of 48,132 records are marked as Delayed even though the actual delivery time is less than the estimated time.
- A further 6,030 records are marked as On Time even though the actual delivery time is greater than or equal to the estimated time.
- These records do not follow the expected time-based classification rule.
- The high number of flagged records suggests that the business definition of Delivery_Status requires clarification before any correction is considered.
- No values were changed because the intended business rule has not yet been independently confirmed.

In [162]:
## Loading and inspecting the data dictionary to understand dataset definitions

data_dictionary = pd.read_csv('/content/data_dictionary.csv')
data_dictionary

,File,Column,Description,Type
0,customers.csv,Customer_ID,Unique customer identifier,Key
1,customers.csv,Customer_Segment,Customer segment assigned by business,Dimension
2,restaurants.csv,Restaurant_ID,Unique restaurant identifier,Key
3,restaurants.csv,Restaurant_Rating,Restaurant rating from customers,Measure
4,orders.csv,Order_ID,Unique order identifier,Key
5,orders.csv,Order_Date,Date the order was placed,Date
6,orders.csv,Order_Status,Delivered or Cancelled,Dimension
7,orders.csv,Food_Amount,Food value before fees and discount,Measure
8,orders.csv,Discount,Discount applied to the order,Measure
9,orders.csv,Final_Amount,Final amount charged after discount and fee,Measure


### ### Observation

- The data dictionary defines Estimated_Delivery_Min as the estimated delivery duration and Actual_Delivery_Min as the actual delivery duration.
- Delivery_Status is described as a dimension containing the categories On Time and Delayed.
- The dictionary does not specify the exact business rule used to assign Delivery_Status.
- The investigation identified records that contradict the expected time-based classification.
- Since the intended business rule could not be confirmed from the available documentation, no changes were made to Delivery_Status.
- The identified inconsistencies have been documented for further business clarification.

In [165]:
## Investigating the number of delivery records associated with each Order_ID
## deliveries.head(5)

delivery_check = deliveries.groupby('Order_ID')['Delivery_ID'].count()
delivery_check[delivery_check > 1]

,Delivery_ID
Order_ID,


### Observation

- Each Order_ID is associated with only one Delivery_ID in the Deliveries table.
- No Order_ID values linked to multiple delivery records were identified.
- Therefore, the relationship between Order_ID and Delivery_ID is consistent with a one-to-one relationship within this dataset.
- No correction was required.

In [166]:
## Analyzing the distribution of Delivery_Status categories

print(deliveries['Delivery_Status'].value_counts())

Delivery_Status
Delayed    48153
On Time     7297
Name: count, dtype: int64


### Observation

- The Deliveries table contains 48,153 records classified as Delayed and 7,297 classified as On Time.
- Approximately 86.84% of deliveries are classified as Delayed, while 13.16% are classified as On Time.
- Delayed deliveries represent the majority of records in the dataset.
- This distribution provides useful context for subsequent delivery-performance analysis.
- The distribution alone does not establish whether the status classifications are correct; the previously identified inconsistencies remain subject to business-rule clarification.

## Conclusion

- The Deliveries table contains 55,450 records and 7 columns, with no missing values identified.
- The data types are appropriate for the respective fields, and no invalid categorical values were identified in the checks performed.
- No duplicate `Delivery_ID` values were found. Each `Delivery_ID` is associated with one `Order_ID`, and every `Order_ID` and `Delivery_Partner_ID` in the Deliveries table has a corresponding record in its respective reference table.
- The numerical validation identified zero values in `Distance_KM` and unusually low values in `Actual_Delivery_Min`, including deliveries recorded as taking only 2 minutes. These values were retained because the available evidence was insufficient to justify a correction.
- `Delivery_Status` contains two categories: `Delayed` and `On Time`. The dataset contains 48,153 Delayed records and 7,297 On Time records.
- The investigation identified records where `Delivery_Status` does not agree with the expected classification based on estimated and actual delivery times. However, the data dictionary does not specify the exact business rule used to assign delivery status.
- No changes were made to the delivery-time or status values without sufficient evidence to justify a correction.
- The identified concerns have been documented for further investigation or business clarification.



The Deliveries table has been validated for structure, missing values, data types, numerical ranges, categorical values, identifier uniqueness, and relationships with the Orders and Delivery Partners tables.

No corrections were required for the checks that passed. However, the zero-distance records, unusually low actual delivery times, and delivery-status inconsistencies remain unresolved and should be considered during subsequent analysis.

Therefore, the table is **ready for further analysis with documented data-quality limitations**. The identified concerns should be considered when interpreting delivery-performance results.

# 11. Reviews Table - Data Cleaning and Validation



## Objective

The Reviews table contains customer feedback and ratings associated with orders. This information can help evaluate customer satisfaction and identify patterns in the customer experience.

The objective of this step is to assess the quality of the Reviews table, identify missing, duplicate, inconsistent, or invalid records, and correct issues only where the correction can be logically justified.


## Validation Approach

The Reviews table will be evaluated for:

- Dataset structure and dimensions
- Data types
- Missing values
- Duplicate records
- Duplicate Review IDs, if available
- Invalid or inconsistent values
- Unusual values requiring further investigation
- Validity of categorical and numerical fields
- Logical relationships between relevant fields
- Relationships with the Orders and Customers tables, where applicable

Each identified issue will be investigated before making any correction. Unusual values will not automatically be treated as errors, and records will not be removed without sufficient justification.

In [167]:
## Loading/reading of the Reviews table
reviews = pd.read_csv('/content/reviews.csv')
reviews.head()

,Review_ID,Order_ID,Rating,Review_Category
0,REV0000001,ORD0052701,4,Food Quality
1,REV0000002,ORD0003459,4,Delivery
2,REV0000003,ORD0056962,4,Overall
3,REV0000004,ORD0005611,1,Delivery
4,REV0000005,ORD0059956,4,Food Quality


In [168]:


reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34379 entries, 0 to 34378
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Review_ID        34379 non-null  object
 1   Order_ID         34379 non-null  object
 2   Rating           34379 non-null  int64 
 3   Review_Category  34379 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.0+ MB


### Observation

- The Reviews table contains 34,379 records and 4 columns.
- All columns contain 34,379 non-null values, indicating that no missing values were identified.
- Review_ID, Order_ID, and Review_Category are stored as object data types, while Rating is stored as an integer.
- The data types appear appropriate for the respective fields.
- The validity of rating values, categorical values, and identifier relationships requires further investigation.

In [169]:
## Checking the statistical summary and valid range of the Rating column
reviews.describe()

,Rating
count,34379.000000
mean,3.863027
std,1.067777
min,0.000000
25%,3.000000
50%,4.000000
75%,5.000000
max,6.000000


### Observation

- The Rating column contains 34,379 non-null numerical values.
- The mean rating is approximately 3.86, and the median rating is 4.
- Ratings range from 0 to 6.
- The minimum and maximum values require further investigation to determine whether they are valid under the intended rating scale.
- No rating values will be corrected or removed until the permitted rating scale is confirmed.

In [173]:
## Checking unique values in the Rating and review-category columns to identify invalid or unexpected ratings

print('Different ratings categories are: ' , reviews['Rating'].unique())
print()
print('No. of different ratings categories are: ' , reviews['Rating'].nunique())
print()
print('Different review category categories are: ' , reviews['Review_Category'].unique())
print()
print('No. of different review category categories are: ' , reviews['Review_Category'].nunique())

Different ratings categories are:  [4 1 5 3 2 0 6]

No. of different ratings categories are:  7

Different review category categories are:  ['Food Quality' 'Delivery' 'Overall' 'Packaging']

No. of different review category categories are:  4


In [175]:
## Analyzing the frequency distribution of customer ratings and review_category

print('Frequency disribution of the Rating column')
print(reviews['Rating'].value_counts())
print()
print('Frequency disribution of the Review_Category column')
print(reviews['Review_Category'].value_counts())


Frequency disribution of the Rating column
Rating
4    14258
5    10496
3     5443
2     2756
1     1406
0       12
6        8
Name: count, dtype: int64

Frequency disribution of the Review_Category column
Review_Category
Food Quality    11972
Overall          8662
Delivery         8547
Packaging        5198
Name: count, dtype: int64


### Observation

- The Rating column contains seven distinct values: 0, 1, 2, 3, 4, 5, and 6.
- Ratings of 0 and 6 occur infrequently, with 12 and 8 records respectively. Their validity requires confirmation against the intended rating scale.
- Ratings of 4 and 5 are the most common, with 14,258 and 10,496 records respectively.
- Review_Category contains four distinct values: Food Quality, Delivery, Overall, and Packaging.
- The category values appear meaningful and consistently formatted, with no obvious invalid categories identified.
- No rating values have been corrected or removed at this stage.

In [176]:
## Checking the data dictionary for the permitted Rating scale
data_dictionary

,File,Column,Description,Type
0,customers.csv,Customer_ID,Unique customer identifier,Key
1,customers.csv,Customer_Segment,Customer segment assigned by business,Dimension
2,restaurants.csv,Restaurant_ID,Unique restaurant identifier,Key
3,restaurants.csv,Restaurant_Rating,Restaurant rating from customers,Measure
4,orders.csv,Order_ID,Unique order identifier,Key
5,orders.csv,Order_Date,Date the order was placed,Date
6,orders.csv,Order_Status,Delivered or Cancelled,Dimension
7,orders.csv,Food_Amount,Food value before fees and discount,Measure
8,orders.csv,Discount,Discount applied to the order,Measure
9,orders.csv,Final_Amount,Final amount charged after discount and fee,Measure


### Observation

- The data dictionary defines Rating as the customer rating for the order.
- The dictionary does not specify the permitted rating scale or minimum and maximum values.
- Ratings of 0 and 6 were identified during the numerical investigation.
- Their validity cannot be conclusively determined from the available documentation.
- No rating values were corrected or removed because the permitted rating scale has not been confirmed.

In [182]:
## Checking Review_ID uniqueness to ensure each review has a unique identifier
## reviews.head()

print('No. of duplicate ID is: ' , reviews['Review_ID'].duplicated().sum())

No. of duplicate ID is:  0


### Observation

- No duplicate Review_ID values were identified.
- Each review record has a unique Review_ID.
- Therefore, Review_ID can be treated as a reliable unique identifier for the Reviews table.
- No correction was required.

In [184]:
## Validating the logical relationship between Review_ID against Order_ID
check_review = reviews.groupby('Review_ID')['Order_ID'].nunique()
check_review[check_review > 1]



,Order_ID
Review_ID,


In [185]:
## Validating the logical relationship between Order_ID against Review_ID
check_review = reviews.groupby('Order_ID')['Review_ID'].nunique()
check_review[check_review > 1]

,Review_ID
Order_ID,


### Observation

- Each Review_ID is associated with only one Order_ID.
- Each Order_ID is associated with at most one distinct Review_ID in the Reviews table.
- No conflicting Review_ID-to-Order_ID or multiple-review-per-order relationships were identified.
- The relationship between Review_ID and Order_ID is consistent within the available dataset.
- No correction was required.

In [188]:
## Validating Order_ID references against the Orders table

print('No. of reviews-Order_ID linked with the Order Table is: ' , reviews['Order_ID'].isin(orders['Order_ID']).sum())

No. of reviews-Order_ID linked with the Order Table is:  34379


### Observation

- All 34,379 Order_ID values in the Reviews table have matching records in the Orders table.
- No unmatched Order_ID references were identified.
- Therefore, the Order_ID references in the Reviews table are valid.
- No correction was required for this relationship.

In [189]:
## Analyzing the distribution of Review_Category values

print(reviews['Review_Category'].value_counts())

Review_Category
Food Quality    11972
Overall          8662
Delivery         8547
Packaging        5198
Name: count, dtype: int64


### Observation

- The Reviews table contains four categories: Food Quality, Overall, Delivery, and Packaging.
- Food Quality accounts for 11,972 reviews, making it the most frequently reviewed category.
- Overall and Delivery account for 8,662 and 8,547 reviews respectively.
- Packaging has the lowest review count, with 5,198 reviews.
- All four categories appear meaningful, and their counts account for all 34,379 records.
- No correction was required based on the category distribution.

In [ ]:
## Confirming Order_ID uniqueness in the Orders table before relying on reference validation

## Conclusion

- The Reviews table contains 34,379 records and 4 columns, with no missing values identified.
- The data types are appropriate for the respective fields, with `Rating` stored as an integer and the identifier and category fields stored as objects.
- The `Rating` column contains values ranging from 0 to 6. Ratings of 0 and 6 occur infrequently, with 12 and 8 records respectively. However, the data dictionary does not specify the permitted rating scale, so these values could not be conclusively classified as invalid.
- Ratings of 4 and 5 are the most common, with 14,258 and 10,496 records respectively.
- The `Review_Category` column contains four distinct categories: Food Quality, Overall, Delivery, and Packaging. No obvious invalid or inconsistently formatted categories were identified.
- No duplicate `Review_ID` values were found, establishing `Review_ID` as a reliable unique identifier for the Reviews table.
- Each `Review_ID` is associated with one `Order_ID`, and no order was found to have multiple distinct review IDs in the dataset.
- All 34,379 `Order_ID` values in the Reviews table have corresponding records in the Orders table. No unmatched order references were identified.
- No values were corrected or removed because no identified issue could be conclusively classified as an error requiring correction.



The Reviews table has been validated for dataset structure, missing values, data types, rating distribution, categorical values, identifier uniqueness, and relationships with the Orders table.

No corrections were required based on the checks performed. However, the permitted rating scale remains unconfirmed because the available data dictionary does not specify the valid minimum and maximum rating values.

Therefore, the Reviews table is **ready for further analysis with a documented data-quality limitation**. The rating values of 0 and 6 should be reviewed if the intended rating scale becomes available. Until then, the existing values will be retained without modification.